In [77]:
import cv2
import torch
import numpy as np
from facenet_pytorch import MTCNN, InceptionResnetV1
import os


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Sử dụng thiết bị:", device)
# 2. KHỞI TẠO MODEL
# MTCNN để trích xuất ảnh mẫu (chỉ lấy 1 khuôn mặt rõ nhất)
mtcnn_enroll = MTCNN(keep_all=False, device=device)

# MTCNN để chạy webcam (lấy tất cả các khuôn mặt xuất hiện trong khung hình)
mtcnn_webcam = MTCNN(keep_all=True, device=device) 

# Mô hình FaceNet
facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)


Sử dụng thiết bị: cuda


In [79]:
for name, module in mtcnn_enroll.pnet.named_modules():
    print(name, module)

 PNet(
  (conv1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
  (prelu1): PReLU(num_parameters=10)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): Conv2d(10, 16, kernel_size=(3, 3), stride=(1, 1))
  (prelu2): PReLU(num_parameters=16)
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (prelu3): PReLU(num_parameters=32)
  (conv4_1): Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1))
  (softmax4_1): Softmax(dim=1)
  (conv4_2): Conv2d(32, 4, kernel_size=(1, 1), stride=(1, 1))
)
conv1 Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
prelu1 PReLU(num_parameters=10)
pool1 MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
conv2 Conv2d(10, 16, kernel_size=(3, 3), stride=(1, 1))
prelu2 PReLU(num_parameters=16)
conv3 Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
prelu3 PReLU(num_parameters=32)
conv4_1 Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1))
softmax4_1 Softmax(dim=1)
conv4_2 Conv2d(32, 4, kernel_siz

In [80]:
for name, module in mtcnn_enroll.rnet.named_modules():
    print(name, module)


 RNet(
  (conv1): Conv2d(3, 28, kernel_size=(3, 3), stride=(1, 1))
  (prelu1): PReLU(num_parameters=28)
  (pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): Conv2d(28, 48, kernel_size=(3, 3), stride=(1, 1))
  (prelu2): PReLU(num_parameters=48)
  (pool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv3): Conv2d(48, 64, kernel_size=(2, 2), stride=(1, 1))
  (prelu3): PReLU(num_parameters=64)
  (dense4): Linear(in_features=576, out_features=128, bias=True)
  (prelu4): PReLU(num_parameters=128)
  (dense5_1): Linear(in_features=128, out_features=2, bias=True)
  (softmax5_1): Softmax(dim=1)
  (dense5_2): Linear(in_features=128, out_features=4, bias=True)
)
conv1 Conv2d(3, 28, kernel_size=(3, 3), stride=(1, 1))
prelu1 PReLU(num_parameters=28)
pool1 MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
conv2 Conv2d(28, 48, kernel_size=(3, 3), stride=(1, 1))
prelu2 PReLU(num_parameters=48)
pool2 Max

In [81]:
for name, module in mtcnn_enroll.onet.named_modules():
    print(name, module)

 ONet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (prelu1): PReLU(num_parameters=32)
  (pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (prelu2): PReLU(num_parameters=64)
  (pool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (prelu3): PReLU(num_parameters=64)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv4): Conv2d(64, 128, kernel_size=(2, 2), stride=(1, 1))
  (prelu4): PReLU(num_parameters=128)
  (dense5): Linear(in_features=1152, out_features=256, bias=True)
  (prelu5): PReLU(num_parameters=256)
  (dense6_1): Linear(in_features=256, out_features=2, bias=True)
  (softmax6_1): Softmax(dim=1)
  (dense6_2): Linear(in_features=256, out_features=4, bias=True)
  (dense6_3): Linear(in_features=256, out_features=10, bias=True)
)
conv1 Conv2d

In [82]:
for name, module in facenet.named_modules():
    print(name, module)

 InceptionResnetV1(
  (conv2d_1a): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2a): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2b): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (maxpool_3a): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2d_3b): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_4a):

In [ ]:
# 3. Lấy embedding
def get_embedding(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None:
            print(" Không mở được ảnh:", img_path)
            return None

        # Chuyển BGR (OpenCV) sang RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # mtcnn_enroll resize 160x160 và chuẩn hóa pixel về [-1, 1]
        face_tensor = mtcnn_enroll(img)
        if face_tensor is None:
            print("Không phát hiện được khuôn mặt:", img_path)
            return None
        
        # Thêm  batch: [3, 160, 160] -> [1, 3, 160, 160]
        face_tensor = face_tensor.unsqueeze(0).to(device)
        # Extract embedding
        
        with torch.no_grad(): 
            embedding = facenet(face_tensor)
            
        return embedding.cpu().numpy()[0]

    except Exception as e:
        print("Lỗi extract embedding:", img_path, e)
        return None


In [84]:
embedding = get_embedding(r"D:\xu_li_anh\lab4\test\18-20\25.jpg")
print(embedding)
print(embedding.shape)

[-5.21990471e-02  2.39132475e-02  1.27666881e-02 -2.59363148e-02
  8.00766349e-02  3.27797141e-03  1.26179366e-03 -5.52890413e-02
 -3.38868834e-02  6.93766922e-02  1.08044758e-01 -9.69155785e-03
 -2.00848244e-02 -8.15019310e-02  5.87206148e-03 -3.42147960e-03
 -8.85713473e-02  2.32066563e-03 -4.53844517e-02  4.60050255e-02
  3.87148522e-02  8.52496177e-03  3.53299864e-02 -2.42961217e-02
 -1.10588409e-01  6.08319826e-02  5.81087507e-02 -4.96995896e-02
 -3.31035890e-02 -5.09258732e-02  5.42908423e-02  3.95129248e-02
 -1.25579219e-02  3.76784876e-02  2.30854750e-02  9.75636207e-03
  1.30398139e-01 -4.34433222e-02 -2.60032015e-03 -3.52958823e-03
  6.10597879e-02 -5.56694195e-02  2.26310417e-02  6.09214343e-02
 -1.50309633e-02 -2.00980846e-02  1.22900922e-02  8.79323408e-02
 -7.24994391e-02 -4.07969356e-02 -6.11926615e-02 -2.03849785e-02
 -1.20029822e-02 -9.73987766e-03  7.08516464e-02  1.18871275e-02
  9.06672888e-03  1.57338043e-03  2.66829338e-02  2.91959047e-02
  2.96964422e-02  4.91883

In [ ]:

known_embeddings = {}

known_dir = r"D:\xu_li_anh\lab4\test"

for person_name in os.listdir(known_dir):
    person_path = os.path.join(known_dir, person_name)

    if not os.path.isdir(person_path):
        continue

    embeddings_list = []

    for filename in os.listdir(person_path):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(person_path, filename)

            emb = get_embedding(img_path)
            if emb is not None:
                embeddings_list.append(emb)

    if len(embeddings_list) > 0:
        known_embeddings[person_name] = embeddings_list

print("Loaded:", list(known_embeddings.keys()))


Không phát hiện được khuôn mặt: D:\xu_li_anh\lab4\test\18-20\6.jpg
Không phát hiện được khuôn mặt: D:\xu_li_anh\lab4\test\18-20\7.jpg
Không phát hiện được khuôn mặt: D:\xu_li_anh\lab4\test\18-20\9.jpg
Loaded: ['18-20', '21-30', '31-40', '41-50', '51-60']


In [ ]:
# 5. COSINE SIMILARITY
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
# 6. WEBCAM REAL-TIME
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Không đọc được camera")
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    boxes, probs = mtcnn_webcam.detect(rgb)

    if boxes is not None:
        for box, prob in zip(boxes, probs):

            if prob is None or prob < 0.90:
                continue

            x1, y1, x2, y2 = map(int, box)

            h, w, _ = frame.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            face = rgb[y1:y2, x1:x2]

            try:
                face = cv2.resize(face, (160, 160))
            except:
                continue

            face = face.astype(np.float32)
            face = (face - 127.5) / 128.0

            face_tensor = torch.tensor(face).permute(2, 0, 1).unsqueeze(0).to(device)

            with torch.no_grad():
                emb = facenet(face_tensor).cpu().numpy()[0]

            # Normalize
            emb = emb / np.linalg.norm(emb)

            # FIX MULTI EMBEDDING MATCHING
            best_name = None
            best_score = -1

            for name, emb_list in known_embeddings.items():
                sims = [cosine_similarity(emb, e) for e in emb_list]

                if len(sims) == 0:
                    continue

                max_sim = max(sims)

                if max_sim > best_score:
                    best_score = max_sim
                    best_name = name

            # Threshold
            if best_score > 0.7:
                label = best_name
                color = (0, 255, 0)
                score = best_score
            else:
                label = "Unknown"
                color = (0, 0, 255)
                score = best_score

            # Draw
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            if label != "Unknown":
                text = f"{label}: {score:.2f}"
            else:
                text = "Unknown"

            cv2.putText(frame, text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()